# Face Recognition Attendance System

A desktop application that takes class attendance using a webcam and face recognition, with a graphical interface built in `tkinter`.

## Objective
Replace manual roll calls with an automatic system that:
- **registers** students by storing a numerical "fingerprint" (embedding) of their face,
- **marks attendance** automatically when a registered face appears in front of the camera,
- **saves images of unknown faces** into a separate dated folder so a teacher can verify them manually later.

## Technologies used
| Library | Purpose |
|---|---|
| `OpenCV (cv2)` | Reads the webcam, draws boxes/labels, saves images |
| `InsightFace` (model `buffalo_l`) | Detects faces and converts each face into a 512-number embedding |
| `NumPy` | Vector maths for comparing embeddings |
| `tkinter` + `Pillow` | The GUI and displaying the live video inside it |
| `csv`, `os`, `datetime` | Attendance file, folders, timestamps |
| `threading`, `queue` | Run face detection in the background so the GUI never freezes |

## How the recognition works
1. **Registration:** the face is detected, converted to an embedding and stored together with the student's name.
2. **Attendance:** every detected face is converted to an embedding and compared with all registered embeddings using **cosine similarity** (1.0 = identical, 0 = unrelated).
3. If the best similarity is at least `THRESHOLD`, the person is **known** and attendance is written to `attendance.csv`. Otherwise the person is **unknown** and the frame is saved for review.
4. After one face is handled, the camera closes and the program returns to the main menu.

## Requirements
- Python 3.9+ with a webcam and a desktop display (run in a **local** Jupyter / VS Code, not Google Colab or a headless server, because the app opens a window).
- Packages are listed in the next cell.

## How to use this notebook
Run the cells **from top to bottom**. The last code cell launches the application window. The notebook cell stays "running" until you close that window.

## Cell 1 - Install the dependencies
Run this once. The line is commented out so the notebook does not reinstall packages every time. Remove the `#` to install.

`onnxruntime` is required by InsightFace to run its neural networks on the CPU. The first time the application runs, InsightFace downloads the `buffalo_l` model files automatically, so an internet connection is needed once.

In [1]:
# Run once, then keep it commented:
# !pip install insightface onnxruntime opencv-python numpy pillow

## Cell 2 - Imports
Each import has a specific job:
- `csv`, `os`, `re`, `sys`, `subprocess` - writing the attendance file, handling folders and filenames, and opening a folder in the file explorer.
- `queue`, `threading`, `time` - background face detection, message passing from the worker thread to the GUI, and timing.
- `tkinter`, `ttk`, `messagebox` - the window, the themed widgets and pop-up dialogs.
- `datetime` - timestamps for attendance rows and for unknown-face filenames.
- `cv2`, `numpy`, `norm` - image/camera handling and the maths for comparing faces.
- `PIL (Image, ImageTk)` - converts OpenCV frames into images that `tkinter` can display.
- `FaceAnalysis` - the InsightFace class that detects faces and produces embeddings.
- `warnings.filterwarnings(...)` - hides harmless `FutureWarning` messages from InsightFace.

In [2]:
import csv
import os
import queue
import re
import subprocess
import sys
import threading
import time
import tkinter as tk
import warnings
from datetime import datetime
from tkinter import messagebox, ttk

import cv2
import numpy as np
from numpy.linalg import norm
from PIL import Image, ImageTk
from insightface.app import FaceAnalysis

# Suppress InsightFace future warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Cell 3 - Settings
All tunable values live in one place so behaviour can be changed without touching the logic.

| Setting | Meaning |
|---|---|
| `REGISTERED_DIR` | Folder holding one photo per registered student (`<name>.jpg`) |
| `UNKNOWN_DIR` | Folder for unknown faces, saved as `unknown_faces/<date>/unknown_<time>.jpg` |
| `CSV_FILE` | The attendance file |
| `THRESHOLD` | Minimum cosine similarity to accept a face as a registered student. Raise it if wrong people get matched, lower it if real students are rejected |
| `MIN_DET_SCORE` | Detections with lower confidence are ignored (removes junk detections) |
| `DETECT_INTERVAL` | Seconds between detection passes (detection is slow on CPU, so it is not run on every frame) |
| `UNKNOWN_COOLDOWN` / `SAME_PERSON_SIM` | Stop the same unknown person from being saved repeatedly |
| `GREEN`, `RED`, `ORANGE` | Box colours in OpenCV's BGR order |

In [3]:
# ----------------------------- Settings -----------------------------
REGISTERED_DIR = "registered_faces"
UNKNOWN_DIR = "unknown_faces"       # unknown_faces/<YYYY-MM-DD>/unknown_<HH-MM-SS>.jpg
CSV_FILE = "attendance.csv"

THRESHOLD = 0.45          # cosine similarity needed to count as a registered face
MIN_DET_SCORE = 0.6       # ignore low-confidence detections
DETECT_INTERVAL = 0.3     # seconds between detection passes
UNKNOWN_COOLDOWN = 60     # don't re-save the same unknown person for this many seconds
SAME_PERSON_SIM = 0.5     # similarity above which two unknown faces are the same person

GREEN = (0, 200, 0)
RED = (0, 0, 255)
ORANGE = (0, 165, 255)

## Cell 4 - Helper functions
Small reusable functions used throughout the program:
- `compute_similarity` - cosine similarity between two embeddings: the dot product divided by the product of their lengths.
- `safe_name` - removes characters such as `/ : * ?` that are illegal in filenames, because the student's name becomes the photo's filename.
- `largest_face` - when a photo contains several faces, picks the biggest one (by bounding-box area).
- `ensure_csv` - creates `attendance.csv` with the header row `Name/ID, Time` if it does not exist yet.
- `read_today_records` - reads the CSV and returns only today's entries, so a student already marked today is not marked again, even after restarting.
- `open_folder` - opens a folder in the system file explorer (Windows, macOS and Linux each use a different command).

In [4]:
def compute_similarity(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b)))


def safe_name(name):
    """Remove characters that are illegal in filenames."""
    return re.sub(r'[\\/:*?"<>|]', "_", name).strip()


def largest_face(faces):
    return max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))


def ensure_csv():
    if not os.path.exists(CSV_FILE) or os.path.getsize(CSV_FILE) == 0:
        with open(CSV_FILE, "w", newline="") as f:
            csv.writer(f).writerow(["Name/ID", "Time"])


def read_today_records():
    """Return [(name, 'HH:MM:SS'), ...] for today, oldest first."""
    today = datetime.now().strftime("%Y-%m-%d")
    records = []
    if not os.path.exists(CSV_FILE):
        return records
    with open(CSV_FILE, newline="") as f:
        reader = csv.reader(f)
        next(reader, None)  # header
        for row in reader:
            if len(row) >= 2 and row[1].startswith(today):
                records.append((row[0], row[1][11:]))
    return records


def open_folder(path):
    os.makedirs(path, exist_ok=True)
    if sys.platform.startswith("win"):
        os.startfile(os.path.abspath(path))
    elif sys.platform == "darwin":
        subprocess.Popen(["open", path])
    else:
        subprocess.Popen(["xdg-open", path])

## Program structure (Cells 5-11)
The application is one class, `AttendanceApp`. To explain it step by step, the class is **split into mixins** - small classes that each hold one group of related methods. In the final cell they are combined, so the result behaves exactly like a single class.

| Mixin | Responsibility |
|---|---|
| `UIMixin` | Creates the screens |
| `ModelMixin` | Loads the AI model and known faces |
| `SessionMixin` | Starts and stops the camera |
| `VideoMixin` | Live video loop and GUI updates |
| `DetectionMixin` | Recognition, attendance, unknown faces |
| `RegistrationMixin` | Saving a new face |

**Important design point - threads.** Face detection takes a fraction of a second. If it ran in the GUI thread, the window and video would freeze. So detection runs in a **background thread**, and the thread never touches the GUI directly. It puts messages into a `queue`, and the GUI thread reads them. This is the safe way to use `tkinter` with threads.

## Cell 5 - `UIMixin`: building the screens
Three screens are created once and then shown or hidden:
- **Home screen** - title, status line and the two buttons *Take Attendance* and *Register New Faces* (disabled until the model has loaded) plus *Quit*.
- **Attendance screen** - live video on the left; on the right a status message, counters, a list of students marked today, a button to open the unknown-faces folder, and the **STOP** button.
- **Registration screen** - live video, a name entry box, the *Capture & Register* button, a status message, the list of registered students and *Back to Menu*.

`_show(frame)` hides all screens and displays the requested one. `_build_ui` also defines the fonts/styles and stores the video labels in `self.video_labels` so the video loop knows where to draw each frame.

In [5]:
class UIMixin:
    """Builds every screen of the window (menu, attendance, registration)."""

    def _build_ui(self):
        style = ttk.Style(self)
        style.configure("Title.TLabel", font=("Segoe UI", 20, "bold"))
        style.configure("Sub.TLabel", font=("Segoe UI", 14, "bold"))
        style.configure("Big.TButton", font=("Segoe UI", 14), padding=14)
        style.configure("Stop.TButton", font=("Segoe UI", 12, "bold"), padding=8)

        self.container = ttk.Frame(self, padding=16)
        self.container.pack()
        self.video_labels = {}

        self._build_home()
        self._build_attendance()
        self._build_register()
        self._show(self.home)

    def _show(self, frame):
        for f in (self.home, self.att, self.reg):
            f.pack_forget()
        frame.pack()

    def _build_home(self):
        self.home = ttk.Frame(self.container)
        ttk.Label(self.home, text="Face Attendance System", style="Title.TLabel").pack(pady=(10, 10))
        self.home_status = ttk.Label(self.home, text="Loading face model, please wait...")
        self.home_status.pack(pady=(0, 20))

        self.btn_attendance = ttk.Button(
            self.home, text="1.  Take Attendance", style="Big.TButton", width=28,
            state="disabled", command=lambda: self.start_mode("attendance"))
        self.btn_attendance.pack(pady=8)

        self.btn_register = ttk.Button(
            self.home, text="2.  Register New Faces", style="Big.TButton", width=28,
            state="disabled", command=lambda: self.start_mode("register"))
        self.btn_register.pack(pady=8)

        ttk.Button(self.home, text="Quit", command=self.on_close).pack(pady=(25, 10))

    def _build_attendance(self):
        self.att = ttk.Frame(self.container)
        video = ttk.Label(self.att)
        video.grid(row=0, column=0, sticky="n")
        self.video_labels["attendance"] = video

        side = ttk.Frame(self.att)
        side.grid(row=0, column=1, sticky="ns", padx=(14, 0))

        ttk.Label(side, text="Taking Attendance", style="Sub.TLabel").pack(anchor="w")
        self.att_status = ttk.Label(side, text="", wraplength=280, justify="left")
        self.att_status.pack(anchor="w", pady=(6, 6))

        self.present_var = tk.StringVar(value="Present today: 0")
        self.unknown_var = tk.StringVar(value="Unknown faces saved this session: 0")
        ttk.Label(side, textvariable=self.present_var).pack(anchor="w")
        ttk.Label(side, textvariable=self.unknown_var).pack(anchor="w", pady=(0, 8))

        ttk.Label(side, text="Marked today (newest first):").pack(anchor="w")
        list_frame = ttk.Frame(side)
        list_frame.pack(fill="both", expand=True)
        scroll = ttk.Scrollbar(list_frame)
        scroll.pack(side="right", fill="y")
        self.att_list = tk.Listbox(list_frame, width=34, height=13, yscrollcommand=scroll.set)
        self.att_list.pack(side="left", fill="both", expand=True)
        scroll.config(command=self.att_list.yview)

        ttk.Button(side, text="Open Unknown Faces Folder",
                   command=lambda: open_folder(UNKNOWN_DIR)).pack(fill="x", pady=(10, 4))
        ttk.Button(side, text="STOP Attendance", style="Stop.TButton",
                   command=self.stop_mode).pack(fill="x", pady=(4, 0))

    def _build_register(self):
        self.reg = ttk.Frame(self.container)
        video = ttk.Label(self.reg)
        video.grid(row=0, column=0, sticky="n")
        self.video_labels["register"] = video

        side = ttk.Frame(self.reg)
        side.grid(row=0, column=1, sticky="ns", padx=(14, 0))

        ttk.Label(side, text="Register New Face", style="Sub.TLabel").pack(anchor="w")
        ttk.Label(side, text="Student name / ID:").pack(anchor="w", pady=(10, 2))
        self.name_var = tk.StringVar()
        entry = ttk.Entry(side, textvariable=self.name_var, width=32)
        entry.pack(anchor="w")
        entry.bind("<Return>", lambda e: self.capture_register())

        ttk.Button(side, text="Capture && Register", command=self.capture_register
                   ).pack(fill="x", pady=(10, 4))
        self.reg_status = ttk.Label(side, text="", wraplength=280, justify="left")
        self.reg_status.pack(anchor="w", pady=(4, 8))

        ttk.Label(side, text="Registered students:").pack(anchor="w")
        list_frame = ttk.Frame(side)
        list_frame.pack(fill="both", expand=True)
        scroll = ttk.Scrollbar(list_frame)
        scroll.pack(side="right", fill="y")
        self.reg_list = tk.Listbox(list_frame, width=34, height=11, yscrollcommand=scroll.set)
        self.reg_list.pack(side="left", fill="both", expand=True)
        scroll.config(command=self.reg_list.yview)

        ttk.Button(side, text="Back to Menu", style="Stop.TButton",
                   command=self.stop_mode).pack(fill="x", pady=(10, 0))

## Cell 6 - `ModelMixin`: loading the face model
Loading InsightFace and processing all saved photos takes several seconds, so it is done in a background thread (`_load_model`) while the window opens immediately.

`_load_model`:
1. creates `FaceAnalysis("buffalo_l")` on the CPU and prepares it with a detection size of 640x640,
2. goes through every image in `registered_faces/`, detects the face and stores its embedding in the dictionary `self.known` as `name -> embedding`,
3. sets `self.model_state` to `"ready"` (or `"error"` with a message if something failed).

`_check_model` runs every 200 ms in the GUI thread. When the model is ready it enables the two menu buttons; `_refresh_home` writes the "Ready. Registered students: N" line.

In [13]:
class ModelMixin:
    """Loads the face model and the registered faces in the background."""

    def _load_model(self):
        try:
            fa = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
            fa.prepare(ctx_id=-1, det_size=(640, 640))  # -1 = CPU

            known = {}
            for filename in sorted(os.listdir(REGISTERED_DIR)):
                if filename.lower().endswith((".png", ".jpg", ".jpeg")):
                    img = cv2.imread(os.path.join(REGISTERED_DIR, filename))
                    if img is None:
                        continue
                    faces = fa.get(img)
                    if faces:
                        known[os.path.splitext(filename)[0]] = largest_face(faces).normed_embedding

            self.fa = fa
            self.known = known
            self.model_state = "ready"
        except Exception as e:  # noqa: BLE001
            self.model_error = str(e)
            self.model_state = "error"

    def _check_model(self):
        if self.model_state == "ready":
            self.btn_attendance.config(state="normal")
            self.btn_register.config(state="normal")
            self._refresh_home()
        elif self.model_state == "error":
            self.home_status.config(text=f"Failed to load face model:\n{self.model_error}")
        else:
            self.after(200, self._check_model)

    def _refresh_home(self):
        self.home_status.config(text=f"Ready.  Registered students: {len(self.known)}")

## Cell 7 - `SessionMixin`: starting and stopping a mode
- `start_mode(mode)` is called by the two menu buttons (`"attendance"` or `"register"`). It opens the webcam (`CAP_DSHOW` on Windows for faster start-up), shows an error if the camera cannot be opened, resets the state for the chosen mode, clears old messages from the queue and starts the video loop. In attendance mode it also reloads today's attendance from the CSV.
- `stop_mode(message)` stops the loop, releases the camera, clears the detection results and returns to the home screen. If a `message` is given (for example *"Attendance marked: Riya"*) it is shown on the menu, which is how the result stays visible after the camera closes. Both menu buttons remain available.
- `on_close` is run when the window's X button is pressed; it makes sure the camera is released before the program exits.

In [7]:
class SessionMixin:
    """Opens the camera for a mode and closes it again."""

    def start_mode(self, mode):
        if self.model_state != "ready" or self.running:
            return

        backend = cv2.CAP_DSHOW if os.name == "nt" else cv2.CAP_ANY
        cap = cv2.VideoCapture(0, backend)
        if not cap.isOpened():
            messagebox.showerror("Camera", "Could not open the webcam.")
            return
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

        self.cap = cap
        self.mode = mode
        self.latest_results = []
        self.last_detect = 0.0
        self.video_labels[mode].configure(image="")

        if mode == "attendance":
            records = read_today_records()
            self.marked = {name for name, _ in records}
            self.att_list.delete(0, "end")
            for name, t in records:
                self.att_list.insert(0, f"{t}   {name}")
            self.unknown_cache = []
            self.unknown_count = 0
            self.present_var.set(f"Present today: {len(self.marked)}")
            self.unknown_var.set("Unknown faces saved this session: 0")
            self.att_status.config(text="Camera running. Press STOP when finished.")
            self._show(self.att)
        else:
            self._refresh_registered_list()
            self.name_var.set("")
            self.reg_status.config(text="Face the camera, type a name, then press Capture.")
            self._show(self.reg)

        # discard leftover messages from a previous session
        while not self.events.empty():
            try:
                self.events.get_nowait()
            except queue.Empty:
                break

        self.finishing = False
        self.running = True
        self._tick()

    def stop_mode(self, message=None):
        self.running = False
        self.mode = None
        if self.tick_job is not None:
            self.after_cancel(self.tick_job)
            self.tick_job = None
        if self.cap is not None:
            self.cap.release()
            self.cap = None
        self.latest_results = []
        self.current_frame = None
        self._refresh_home()
        if message:
            self.home_status.config(
                text=f"{message}\nRegistered students: {len(self.known)}")
        self._show(self.home)

    def on_close(self):
        if self.running:
            self.stop_mode()
        self.destroy()

## Cell 8 - `VideoMixin`: the live video loop
`_tick` runs about every 15 ms and does the following each time:
1. reads one frame from the webcam,
2. processes pending messages from the worker thread (`_drain_events`),
3. if no detection is running and `DETECT_INTERVAL` has passed, starts a new background detection on a **copy** of the frame,
4. draws the latest boxes and labels on the frame, converts it from BGR to RGB and shows it in the window.

The `finishing` flag stops new detections from starting once a face has been handled, and the `if not self.running: return` check prevents the loop from reading a camera that was just released.

`_drain_events` handles the messages: `marked` (add to the on-screen list), `unknown` (update the counter), `finished` (close the camera with a result message) and `error`.

In [8]:
class VideoMixin:
    """Shows the live video and receives messages from the worker thread."""

    def _tick(self):
        if not self.running:
            return

        ok, frame = self.cap.read()
        if not ok:
            self.stop_mode()
            messagebox.showerror("Camera", "Lost connection to the webcam.")
            return
        self.current_frame = frame
        self._drain_events()
        if not self.running:   # a "finished" event just closed the camera
            return

        now = time.time()
        if (not self.busy and not self.finishing
                and now - self.last_detect >= DETECT_INTERVAL):
            self.busy = True
            self.last_detect = now
            threading.Thread(target=self._detect_worker,
                             args=(frame.copy(), self.mode), daemon=True).start()

        display = frame.copy()
        for (x1, y1, x2, y2), label, color in self.latest_results:
            cv2.rectangle(display, (x1, y1), (x2, y2), color, 2)
            if label:
                cv2.putText(display, label, (x1, max(y1 - 10, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        rgb = cv2.cvtColor(display, cv2.COLOR_BGR2RGB)
        self.photo = ImageTk.PhotoImage(Image.fromarray(rgb))
        self.video_labels[self.mode].configure(image=self.photo)

        self.tick_job = self.after(15, self._tick)

    def _drain_events(self):
        while True:
            try:
                event = self.events.get_nowait()
            except queue.Empty:
                break
            kind = event[0]
            if kind == "marked":
                _, name, t = event
                self.att_list.insert(0, f"{t}   {name}")
                self.present_var.set(f"Present today: {len(self.marked)}")
                self.att_status.config(text=f"Attendance marked: {name}")
            elif kind == "unknown":
                self.unknown_count += 1
                self.unknown_var.set(f"Unknown faces saved this session: {self.unknown_count}")
                self.att_status.config(text=f"Unknown face saved: {os.path.basename(event[1])}")
            elif kind == "finished":
                if self.running and self.mode == "attendance":
                    self.stop_mode(event[1])
            elif kind == "error":
                label = self.att_status if self.mode == "attendance" else self.reg_status
                label.config(text=f"Error: {event[1]}")

## Cell 9 - `DetectionMixin`: recognition and attendance
This is the core logic.
- `_detect_worker` (background thread): runs InsightFace on the frame under a lock (the model must not be used by two threads at once), discards low-confidence faces, then either processes attendance or, in registration mode, just prepares preview boxes (green = exactly one face, orange = none or several).
- `_match` compares a face embedding with every registered embedding and returns the best name only if its similarity is at least `THRESHOLD`.
- `_process_attendance` goes through each detected face:
  - **known and not yet marked** -> a row `name, timestamp` is appended to `attendance.csv`,
  - **known but already marked** -> nothing is written, and the message says so,
  - **unknown** -> `_handle_unknown` is called.
  Afterwards it sends the `finished` message so the camera closes.
- `_handle_unknown` saves the whole frame, with a red box around the unknown face, to `unknown_faces/<date>/unknown_<time>.jpg` for later manual verification. A short memory of recent unknown embeddings prevents the same person being saved many times.

In [9]:
class DetectionMixin:
    """Face detection, matching, attendance marking and unknown-face saving."""

    def _detect_worker(self, frame, mode):
        try:
            with self.model_lock:
                faces = [f for f in self.fa.get(frame) if f.det_score >= MIN_DET_SCORE]

            if mode != self.mode or not self.running:
                return  # user already left this mode

            if mode == "attendance":
                self.latest_results = self._process_attendance(frame, faces)
            else:
                color = GREEN if len(faces) == 1 else ORANGE
                self.latest_results = [(f.bbox.astype(int), "", color) for f in faces]
        except Exception as e:  # noqa: BLE001
            self.events.put(("error", str(e)))
        finally:
            self.busy = False

    def _match(self, embedding):
        best_name, best_sim = None, -1.0
        for name, known_emb in self.known.items():
            sim = compute_similarity(embedding, known_emb)
            if sim > best_sim:
                best_name, best_sim = name, sim
        if best_sim >= THRESHOLD:
            return best_name, best_sim
        return None, best_sim

    def _process_attendance(self, frame, faces):
        results = []
        messages = []
        now = datetime.now()
        for face in faces:
            box = face.bbox.astype(int)
            name, sim = self._match(face.normed_embedding)

            if name is None:
                results.append((box, "Unknown", RED))
                self._handle_unknown(frame, face, box, now)
                messages.append("Unknown face saved for review")
            else:
                results.append((box, f"{name} ({sim:.2f})", GREEN))
                if name not in self.marked:
                    stamp = now.strftime("%Y-%m-%d %H:%M:%S")
                    with open(CSV_FILE, "a", newline="") as f:
                        csv.writer(f).writerow([name, stamp])
                    self.marked.add(name)
                    self.events.put(("marked", name, stamp[11:]))
                    messages.append(f"Attendance marked: {name}")
                else:
                    messages.append(f"{name} was already marked today")

        # A face was handled: tell the GUI thread to close the camera
        if messages:
            self.finishing = True
            self.events.put(("finished", "; ".join(messages)))
        return results

    def _handle_unknown(self, frame, face, box, now):
        """Save the frame for manual review, once per unknown person per cooldown."""
        t = time.time()
        emb = face.normed_embedding

        self.unknown_cache = [c for c in self.unknown_cache if t - c[1] < UNKNOWN_COOLDOWN]
        for cached in self.unknown_cache:
            if compute_similarity(emb, cached[0]) >= SAME_PERSON_SIM:
                cached[1] = t   # still the same person in front of the camera
                return
        self.unknown_cache.append([emb, t])

        folder = os.path.join(UNKNOWN_DIR, now.strftime("%Y-%m-%d"))
        os.makedirs(folder, exist_ok=True)
        base = now.strftime("unknown_%H-%M-%S")
        path = os.path.join(folder, base + ".jpg")
        i = 1
        while os.path.exists(path):
            path = os.path.join(folder, f"{base}_{i}.jpg")
            i += 1

        img = frame.copy()
        x1, y1, x2, y2 = box
        cv2.rectangle(img, (x1, y1), (x2, y2), RED, 2)
        cv2.imwrite(path, img)
        self.events.put(("unknown", path))

## Cell 10 - `RegistrationMixin`: registering a new student
`capture_register` runs when the user presses *Capture & Register*. It:
1. checks that a name was typed and cleans it with `safe_name`,
2. detects faces in the current frame and requires **exactly one** face,
3. warns if this face looks like a student already registered under a different name (possible duplicate), and asks before overwriting an existing name,
4. saves the photo to `registered_faces/<name>.jpg` and stores the embedding in `self.known`, so the student can be recognised immediately without restarting,
5. refreshes the on-screen list and shows a success message.

`_refresh_registered_list` redraws the list of registered names in alphabetical order.

In [10]:
class RegistrationMixin:
    """Registers a new face from the live camera frame."""

    def _refresh_registered_list(self):
        self.reg_list.delete(0, "end")
        for name in sorted(self.known, key=str.lower):
            self.reg_list.insert("end", name)

    def capture_register(self):
        if self.mode != "register" or self.current_frame is None:
            return

        name = safe_name(self.name_var.get())
        if not name:
            self.reg_status.config(text="Please enter a name / ID first.")
            return

        frame = self.current_frame.copy()
        with self.model_lock:
            faces = [f for f in self.fa.get(frame) if f.det_score >= MIN_DET_SCORE]

        if len(faces) == 0:
            self.reg_status.config(text="No face detected. Try again.")
            return
        if len(faces) > 1:
            self.reg_status.config(text="Multiple faces detected. Only one person at a time.")
            return

        embedding = faces[0].normed_embedding

        match_name, sim = self._match(embedding)
        if match_name and match_name != name:
            if not messagebox.askyesno(
                    "Possible duplicate",
                    f"This face looks like the already registered '{match_name}' "
                    f"(similarity {sim:.2f}).\n\nRegister as '{name}' anyway?"):
                return
        if name in self.known:
            if not messagebox.askyesno("Overwrite", f"'{name}' is already registered. Replace?"):
                return

        if not cv2.imwrite(os.path.join(REGISTERED_DIR, f"{name}.jpg"), frame):
            self.reg_status.config(text="Could not save the image. Try a different name.")
            return

        self.known[name] = embedding
        self.name_var.set("")
        self._refresh_registered_list()
        self.reg_status.config(text=f"Successfully registered {name}!")

## Cell 11 - `AttendanceApp`: putting the class together
`AttendanceApp` inherits from all six mixins and from `tk.Tk` (the main window). Its `__init__`:
- creates the folders and the CSV header if they do not exist,
- defines the shared state: the model and known faces, the lock, the message queue, camera and loop variables,
- builds the screens (`_build_ui`),
- starts the model-loading thread and begins checking whether it has finished.

After this cell the application is fully defined, but nothing is shown yet.

In [11]:
class AttendanceApp(UIMixin, ModelMixin, SessionMixin, VideoMixin,
                    DetectionMixin, RegistrationMixin, tk.Tk):
    """The complete application: all mixins combined with the Tk window."""

    def __init__(self):
        super().__init__()
        self.title("Face Attendance System")
        self.resizable(False, False)
        self.protocol("WM_DELETE_WINDOW", self.on_close)

        os.makedirs(REGISTERED_DIR, exist_ok=True)
        os.makedirs(UNKNOWN_DIR, exist_ok=True)
        ensure_csv()

        # model / data
        self.fa = None
        self.known = {}                 # name -> embedding
        self.model_state = "loading"    # loading | ready | error
        self.model_error = ""
        self.model_lock = threading.Lock()
        self.events = queue.Queue()     # worker thread -> GUI thread

        # runtime state
        self.mode = None                # "attendance" | "register" | None
        self.cap = None
        self.running = False
        self.busy = False               # a detection worker is running
        self.finishing = False          # a face was handled; camera is about to close
        self.current_frame = None
        self.latest_results = []        # [(box, label, color)]
        self.last_detect = 0.0
        self.marked = set()
        self.unknown_cache = []         # [[embedding, last_seen_time], ...]
        self.unknown_count = 0
        self.tick_job = None
        self.photo = None

        self._build_ui()
        threading.Thread(target=self._load_model, daemon=True).start()
        self.after(200, self._check_model)

## Cell 12 - Launch the application
Creates the window and starts `mainloop()`. Choose:
- **Register New Faces** first - enter a name, face the camera and press *Capture & Register*. Repeat for each student.
- **Take Attendance** - the camera opens, recognises the first face, writes the result and closes. The menu then shows what happened.

This cell keeps running until the window is closed; this is normal for a GUI program. To start again, run the cell again.

In [12]:
app = AttendanceApp()
app.mainloop()

## Output files and folders
| Path | Content |
|---|---|
| `registered_faces/<name>.jpg` | One photo per registered student (re-read at every start) |
| `attendance.csv` | Columns `Name/ID`, `Time` - one row per student per day |
| `unknown_faces/<date>/unknown_<time>.jpg` | Frames with unrecognised faces, outlined in red, for manual verification |

## Testing checklist
1. Register at least two people; check that photos appear in `registered_faces/`.
2. Take attendance with a registered person; check the new row in `attendance.csv` and that the camera closes.
3. Take attendance again with the same person; the menu should say they were already marked today and no new row should be added.
4. Take attendance with someone who is not registered; check the new image in `unknown_faces/<today>/`.
5. Try registering with no face, two faces, an empty name, and a duplicate name to see the error messages.

## Limitations and possible improvements
- A photo or video of a person held up to the camera can fool the system. **Liveness / anti-spoofing** detection would fix this.
- Each *Take Attendance* press records one face. A continuous mode that runs until a stop button is pressed (as in an earlier version) suits whole-class marking better.
- Recognition accuracy depends on lighting and camera quality. Registering several photos per student would improve it.
- Everything runs on the CPU. With a GPU, `onnxruntime-gpu` and `CUDAExecutionProvider` would make detection faster.
- Attendance is stored in a CSV file. A database would be better for many classes and for reports.

## Conclusion
The project combines a deep-learning face model (InsightFace), computer vision (OpenCV) and a GUI (`tkinter`) into a practical attendance tool. Registered students are recognised automatically, duplicates are avoided, and unknown faces are saved for a teacher to check.